Natural language inference development for personal project.  
Problem scope:  
- mass ingestion of information daily leads to superficial understanding of ideas, concepts, and real-world events
- developing model to score output in own words after morning research (facts, not opinions)
  - v1: Facebook's BART model (pretrained on MNLI dataset)
- final model is a node within a langgraph framework


In [1]:
# dependencies + hf token
!pip install --upgrade transformers
from huggingface_hub import login
from google.colab import userdata

hf_token = userdata.get('hf_token')

In [3]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large-mnli")
model = AutoModelForSequenceClassification.from_pretrained("facebook/bart-large-mnli")

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

In [4]:
model.eval()
# test -> [0: contradiction, 1: neutral, 2: entailment] -> argmax on logits to find class
inputs = tokenizer(
    'I am interested in Artificial Intelligence.',
    'Machine Learning is boring!',
    return_tensors='pt',
    padding=True, # potential problem with ingested newsletter vs short summary returned
    truncation=True
)
outputs = model(**inputs)
predicted_class_id = outputs.logits.argmax(axis=1).item()
print(f"ID: {predicted_class_id}")

ID: 0


Truncation argument is problematic for large ingested news sources. Max input length for model is 512 tokens, if ingests 2000 then 1488 are cut off. Using summary model to find core semantic meaning of essay input for premise field may be solution.

In [8]:
# summarization model
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer_summ = AutoTokenizer.from_pretrained("facebook/bart-large-cnn")
model_summ = AutoModelForSeq2SeqLM.from_pretrained("facebook/bart-large-cnn")

ARTICLE="In a post on Truth Social, Trump said Lebanese President Joseph Aoun and Israeli Prime Minister Benjamin Netanyahu have agreed that in order to achieve PEACE between their Countries, they will formally begin a 10 Day CEASEFIRE at 5 P.M. EST Trump said that he had directed Vice President JD Vance and Secretary of State Rubio to work with the countries toward achieving a Lasting PEACE, adding that he had invited Aoun and Netanyahu to take part in peace talks at the White House. Lebanese Prime Minister Nawaf Salam welcomed the ceasefire in a post on X, saying it had been a key objective for Lebanon in talks this week. In a video statement, Netanyahu confirmed he had agreed to a temporary ceasefire, but said that Israel had not agreed to withdraw from southern Lebanon, a key demand of Hezbollah, adding that the group must be dismantled. We are remaining in Lebanon in an expanded security zone, he said, adding that this was necessary due to the danger of an invasion and to prevent fire into Israel. It was unclear when or if those displaced from their homes in southern Lebanon by Israel's invasion would be allowed to return. Lebanon's Parliament Speaker Nabih Berri warned people to postpone their return to their towns and villages until the situation becomes clearer, in accordance with the ceasefire agreement."

inputs_summ = tokenizer_summ(ARTICLE, max_length=1024, truncation=True, return_tensors="pt")

# use generate method
summary_ids = model_summ.generate(
    inputs_summ["input_ids"],
    num_beams=4,
    min_length=100,
    max_length=130,
    early_stopping=True
)

# decode for print
summary = tokenizer_summ.decode(summary_ids[0], skip_special_tokens=True)

print(summary)


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

Lebanese President Joseph Aoun and Israeli Prime Minister Benjamin Netanyahu have agreed to a 10 Day ceasefire. Trump said he had directed Vice President JD Vance and Secretary of State Rubio to work with the countries toward achieving a Lasting PEACE. Netanyahu confirmed he had agreed to the temporary ceasefire, but said that Israel had not agreed to withdraw from southern Lebanon, a key demand of Hezbollah. It was unclear when or if those displaced from their homes in southern Lebanon by Israel's invasion would be allowed to return.


Design for langgraph (v1):
- build pipeline utilizing summarization and nli for meaning comparison
- content retrieval > user & summarization model (summarization_input)
  - user generates own thoughts (call this user_input)
- nli model(premise=summarization_input, argument=user_input)

- worried about latency with model downloads